# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

2. **featuretools** - Automated feature generation via Deep Feature Synthesis

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data from Databricks Catalog

In [0]:
from backend.core.utils import process_col_names

# Read from Databricks Unity Catalog volume
# df = spark.read.csv(
#     "/Volumes/aidetic_databricks/default/credit_card_transactions/credit_card_transactions.csv",
#     header=True,
#     inferSchema=True
# )

# df = df.drop("Unnamed: 0")

CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"

# # Transaction Data
# TABLE_NAME = "credit_card_transactions"

# # Bank Customers
# TABLE_NAME = "hdfc_demo_bank_customers"

# # Bank Transactions
# TABLE_NAME = "hdfc_demo_bank_transactions"

TABLE_NAMES = ["hdfc_demo_bank_customers", "hdfc_demo_bank_transactions"]

tables = {}
for _name in TABLE_NAMES:
  credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{_name}"
  print(credit_card_transactions_table_name)

  # credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"

  tables[_name] = spark.read.table(credit_card_transactions_table_name)

  print("TABLE NAME: ", _name)
  print(f"Dataset shape: {tables[_name].count():,} rows x {len(tables[_name].columns)} columns")
  tables[_name].printSchema()

In [0]:
display(tables[TABLE_NAMES[0]].limit(5))

In [0]:
# # For Credit Card Transactions
# index_col = "trans_num"
# target_entity="is_fraud"
# time_index="trans_date_trans_time"

# For Bank Customers
index_col = "customer_id"
target_entity="responded"
time_index="contact_timestamp"

# # For Bank Transactions
# index_col = "cus"
# target_entity="responded"
# time_index="contact_timestamp"

---
# Featuretools - Automated Feature Generation

Use Deep Feature Synthesis (DFS) to automatically discover and generate
features from the transaction data. Featuretools creates transform and
aggregation features based on the entity relationships.

In [0]:
!pip install featuretools

## Multi Table Implementation

In [0]:
from backend.core.features.featuretools_engine import FeaturetoolsEngine, EntityConfig, RelationshipConfig

ft_engine = FeaturetoolsEngine(
    spark=spark,
    max_rows_for_pandas=None,
    verbose=True
)

entities = {
            'customers': EntityConfig('customers', 'customer_id', time_index='contact_timestamp'),
            'transactions': EntityConfig('transactions', 'transaction_id', time_index='transaction_date')
        }

relationships = [
            RelationshipConfig('customers', 'customer_id', 'transactions', 'customer_id')
        ]

In [0]:
ft_result = ft_engine.run_dfs_multi_table(
            dataframes={'customers': tables["hdfc_demo_bank_customers"], 'transactions': tables["hdfc_demo_bank_transactions"]},
            entities=entities,
            relationships=relationships,
            target_entity='customers'
        )

print(tables["hdfc_demo_bank_customers"].count(), tables["hdfc_demo_bank_transactions"].count())
print(ft_result.feature_matrix.shape)

In [0]:
ft_result.feature_matrix.head()

## Single Table Implementation

In [0]:
# from backend.core.features.featuretools_engine import FeaturetoolsEngine

# ft_engine = FeaturetoolsEngine(
#     spark=spark,
#     max_rows_for_pandas=None,
#     verbose=True
# )

# # Run DFS on a single table
# # For Credit Card Transactions
# # Use trans_num as the unique index column
# ft_result = ft_engine.run_dfs_single_table(
#     spark_df=df,
#     # index_col="trans_num", # For Credit Card Transactions
#     index_col=index_col,
#     # target_entity=target_entity,
#     # time_index="trans_date_trans_time", # For Credit Card Transactions
#     time_index=time_index,
#     max_depth=3,
#     trans_primitives=["add_numeric", "subtract_numeric", "multiply_numeric"],
#     max_features=50
# )



In [0]:
# List all columns to drop from the main DataFrame
# _cols_meta = ["trans_num", "trans_date_trans_time"] # For Credit Card Transactions

df = tables["hdfc_demo_bank_customers"] #ONly for Bank Customers
_cols_meta = [index_col, time_index]

col_to_drop = df.columns
# print("Columns to Drop: ", col_to_drop, type(col_to_drop))
for _col in _cols_meta:
    # print(_col)
    if _col in col_to_drop: col_to_drop.remove(_col)

print("Columns to Drop: ", col_to_drop, type(col_to_drop))

feature_names = []
for _name in ft_result.feature_matrix.columns:
    feature_names.append(_name.lower().replace("(", "_").replace(")", "_").replace(".", "_").replace(" ", "_"))

ft_result.feature_matrix.columns = feature_names

print(f"\nFEATURETOOLS RESULTS:")
print(f"  Features generated: {len(ft_result.feature_names)}")
print(f"  Generation time: {ft_result.generation_time:.2f}s")
print(f"\nSample generated feature names:")
for name in ft_result.feature_names[:10]:
    print(f"    - {name}")

# Merge featuretools features back into the main DataFrame
df = ft_engine.to_spark(ft_result.feature_matrix, df, join_column=index_col, )

print(f"\nDataFrame after featuretools merge: {len(df.columns)} columns")
print(f"Number of rows: {df.count()}")

In [0]:
# Get human-readable feature descriptions
descriptions = ft_engine.get_feature_descriptions()

print("FEATURE DESCRIPTIONS (first 10):")
print("-" * 60)
for desc in descriptions[:10]:
    print(f"  {desc['name']}: {desc['type']}")

In [0]:
# Preview the feature matrix
ft_result.feature_matrix.head()

In [0]:
print("Data with duplicated columns: ", len(df.columns))

# Fix: Remove duplicate columns by renaming all columns to unique names first
col_names = df.columns
seen = {}
unique_names = []
for col in col_names:
    if col not in seen:
        seen[col] = 0
        unique_names.append(col)
    else:
        seen[col] += 1
        unique_names.append(f"{col}_dup{seen[col]}")

# Rename all columns to unique names
df = df.toDF(*unique_names)

for c in df.columns:
    df = df.withColumnRenamed(c, c.replace(" + ", "_sum_"))


#Drop original columns
df = df.drop(*col_to_drop)

print("Data with deduplicated columns: ",len(df.columns))
print(df.count())
display(df.limit(5))

In [0]:
print(df.printSchema())


## Feature Table

Saving data in feature tables

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()


In [0]:
CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "feature_store"
TABLE_NAME = "hdfc_demo_customer_transaction_featuretools"

# credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.hdfc_demo_credit_card_trans_featuretools"
credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# Write to tables in Unity Catalog
# spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")




In [0]:
# Create feature table with `trans_num` as the primary key.
# Take schema from DataFrame output by featuretool_features
customer_feature_table = fe.create_table(
  name=credit_card_transactions_table_name,
  # primary_keys=['trans_num', 'trans_date_trans_time'],
  # primary_keys=_cols_meta,
  primary_keys=index_col,
  # timeseries_columns='trans_date_trans_time',
  # timeseries_columns=time_index,
  schema=df.schema,
  description='Transaction features'
)

In [0]:
fe.write_table(
  name=credit_card_transactions_table_name,
  df = df,
  mode = 'merge'
)